# LigandMPNN Inverse Folding

![LigandMPNN Inverse Folding](https://proto-bio.github.io/proto-assets/images/tool/ligandmpnn/hero.png)

This notebook demonstrates how to use `run_ligandmpnn_sample` to design protein sequences conditioned on both backbone structure and nearby ligand atoms, and `run_ligandmpnn_score` to score sequence-structure compatibility in the same context. LigandMPNN extends ProteinMPNN by incorporating small molecule geometry into sequence design, enabling the creation of binding pocket sequences that are compatible with a specific ligand conformation. When no ligand atoms are present in the input structure, LigandMPNN behaves similarly to ProteinMPNN.


In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("ligandmpnn")
display_overview("ligandmpnn")
display_docs_section("ligandmpnn", "Background")

# LigandMPNN

Released in 2023, LigandMPNN is an inverse-folding model that designs a sequence for a protein backbone while explicitly accounting for the non-protein atoms around it: small-molecule ligands, nucleotides, and metal ions. It extends ProteinMPNN, which ignores those atoms, and substantially improves sequence recovery for residues that contact ligands, nucleic acids, or metals, making it a model of choice for binding-site and cofactor-aware design. It can design sequences for a backbone and score how well a sequence fits a structure.

LigandMPNN ([Dauparas et al., 2025](https://doi.org/10.1038/s41592-025-02626-1)) solves the inverse-folding problem for biomolecular assemblies: given a protein backbone together with the non-protein atoms around it, it predicts an amino-acid sequence compatible with that environment. It is a direct extension of ProteinMPNN, which sees only protein backbone atoms and is therefore blind to the bound ligands, nucleic acids, and metals that strongly shape which residues fit.

Internally, LigandMPNN keeps ProteinMPNN's message-passing design model and adds a second graph over the non-protein atoms. Residues and nearby ligand atoms exchange messages, and the model reads each atom's chemical element, which is what lets it reason about coordinating a metal or packing against a large or unusual ligand. It generates the sequence autoregressively and can also produce sidechain conformations so binding interactions can be inspected directly. On native backbones it recovers roughly 63% of the native residues that contact small molecules, 51% of those contacting nucleotides, and 78% of those coordinating metals.

The reference implementation is maintained by the [Institute for Protein Design](https://www.ipd.uw.edu/) at [dauparas/LigandMPNN](https://github.com/dauparas/LigandMPNN).

## Available tools

In [2]:
display_available_tools("ligandmpnn")

- **`run_ligandmpnn_sample()`** — Sample protein sequences using LigandMPNN
- **`run_ligandmpnn_score()`** — Score protein sequences using LigandMPNN

### `run_ligandmpnn_sample`

LigandMPNN analyzes the backbone coordinates and any ligand atoms present in the structure to generate sequences optimized for the target fold and binding environment. We use GFP as a minimal runnable example. For ligand-aware design, provide a structure file that includes HETATM records for the bound ligand of interest. You can optionally fix residue positions to preserve critical interactions and exclude specific amino acids from the designed sequences.

In [3]:
from proto_tools import (
    ESMFoldInput,
    InverseFoldingStructureInput,
    LigandMPNNSampleConfig,
    LigandMPNNSampleInput,
    LigandMPNNScoringConfig,
    LigandMPNNScoringInput,
    SequenceStructurePair,
    Complex,
    run_esmfold,
    run_ligandmpnn_sample,
    run_ligandmpnn_score,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [4]:
# Display input docs
display_api_reference("ligandmpnn", "input", "run_ligandmpnn_sample")

# Load GFP structure and configure design constraints
gfp_structure = get_gfp_structure()
struct_input = InverseFoldingStructureInput(
    structure=gfp_structure,
    fixed_positions={"A": [2, 3, 4]},
)
inputs = LigandMPNNSampleInput(inputs=[struct_input])

**Input** — `InverseFoldingInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>inputs</code> | <code>list[proto_tools.tools.inverse_folding.shared_data_models.InverseFoldingStructureInput]</code> | required | Per-structure inputs, each containing a structure and optional selections. |

In [5]:
# Display config docs
display_api_reference("ligandmpnn", "config", "run_ligandmpnn_sample")

# Configure sampling | see docs above for all fields
config = LigandMPNNSampleConfig(
    num_sequences_per_structure=2,  # Generate 2 sequence complexes
    temperature=0.1,                # Conservative complexes
    excluded_amino_acids=["C"],     # Exclude cysteine
    seed=42,
    device="cuda",                  # Change to "cpu" if no GPU available
)

**Config** — `LigandMPNNSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on. Options include 'cuda' (NVIDIA GPU), 'cpu' (CPU execution) |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>num_sequences_per_structure</code> | <code>int</code> | <code>1</code> | Total number of sequences to generate per input structure. |
| <code>batch_size</code> | <code>int &#124; None</code> | <code>None</code> | Number of sequences to process simultaneously on GPU. Defaults to num_sequences_per_structure. |
| <code>temperature</code> | <code>float</code> | <code>0.1</code> | Sampling temperature; lower = greedier, higher = more diverse |
| <code>model_type</code> | <code>Literal['ligand_mpnn']</code> | <code>'ligand_mpnn'</code> | LigandMPNN model variant (ligand-aware weights). |
| <code>ligand_mpnn_use_atom_context</code> | <code>bool</code> | <code>True</code> | Encode ligand atom context in the message-passing graph |
| <code>ligand_mpnn_use_side_chain_context</code> | <code>bool</code> | <code>False</code> | Condition on sidechain atoms of fixed residues |
| <code>ligand_mpnn_cutoff_for_score</code> | <code>float</code> | <code>8.0</code> | Ligand-residue distance cutoff (A) for interface recovery score |
| <code>excluded_amino_acids</code> | <code>list[Literal['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']] &#124; None</code> | <code>None</code> | Single-letter codes of amino acids to exclude (e.g. ['C'] to forbid cysteine) |

In [6]:
# Run the sampling function
result = run_ligandmpnn_sample(inputs, config)

LigandMPNN sampling:   0%|          | 0/1 [00:00<?, ?structure/s]

In [7]:
# Display output docs
display_api_reference("ligandmpnn", "output", "run_ligandmpnn_sample")

# Each input structure yields a LigandMPNNDesignSet of LigandMPNNDesign complexes.
# A LigandMPNNDesign holds every chain of the input (designed + fixed context)
# plus per-design recovery metrics.
import math

for i, design_set in enumerate(result.design_sets):
    print(f"Structure {i}: {len(design_set.complexes)} complexes")
    for j, design in enumerate(design_set.complexes, 1):
        recovery = design.metrics["sequence_recovery"]
        interface = design.metrics.get("ligand_interface_sequence_recovery", float("nan"))
        iface_str = "N/A (no ligand interface)" if math.isnan(interface) else f"{interface:.3f}"
        for chain, was_designed in zip(design.chains, design.designed, strict=True):
            tag = "designed" if was_designed else "fixed context"
            preview = f"{chain.sequence[:50]}..." if len(chain.sequence) > 50 else chain.sequence
            print(f"  Design {j} chain {chain.id} ({tag}, {len(chain.sequence)} aa): {preview}")
        print(f"  Design {j} metrics: recovery={recovery:.3f} | interface={iface_str}")

**Output** — `LigandMPNNSampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>design_sets</code> | <code>list[proto_tools.tools.inverse_folding.ligandmpnn.ligandmpnn_sample.LigandMPNNDesignSet]</code> | required | One LigandMPNNDesignSet per input structure, in input order. |

Structure 0: 2 complexes
  Design 1 chain A (designed, 227 aa): SKGAELMKGEVPVKVTLEGDVNGKKFKVEGEGHVRAQEGLLELHFECTSG...
  Design 1 metrics: recovery=0.527 | interface=0.538
  Design 2 chain A (designed, 227 aa): SKGAELLKGRVPVRVHLEGDVNGKKFKVEGEGHGDASRGLLRLHFRCTSG...
  Design 2 metrics: recovery=0.545 | interface=0.577


### Fold-validating a design

A `LigandMPNNDesign` already carries every chain of the complex, so it can be handed straight to a structure predictor as `design` to check that the designed sequence folds back to the target without manually reassembling chains.

In [8]:
# A LigandMPNNDesign drops straight into a structure predictor.
design = result.design_sets[0].complexes[0]
fold = run_esmfold(ESMFoldInput(complexes=[design]))
print(f"Folded {len(design.chains)} chains | success={fold.success}")

Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

Folded 1 chains | success=True


### `run_ligandmpnn_score`

Use `run_ligandmpnn_score` to compute LigandMPNN log-likelihood and perplexity for a sequence in a structure context. This example loads a small protein-only backbone (the in-tree `example_input_fixture.pdb`) and scores its native chain-A sequence — LigandMPNN's score path requires a fully-canonical polymer chain, so chromophore / non-standard residues stay on the sampling side demonstrated above.


In [9]:
display_api_reference("ligandmpnn", "input", "run_ligandmpnn_score")
display_api_reference("ligandmpnn", "config", "run_ligandmpnn_score")
display_api_reference("ligandmpnn", "output", "run_ligandmpnn_score")

# Resolve the in-tree clean-protein fixture by walking up to the repo root.
from pathlib import Path
from proto_tools.entities.structures import Structure
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
score_structure = Structure.from_file(str(_repo_root / "proto_tools" / "tools" / "inverse_folding" / "example_input_fixture.pdb"))
score_sequence = score_structure.get_chain_sequence("A", remove_non_standard=True)

score_inputs = LigandMPNNScoringInput(
    sequence_structure_pairs=[SequenceStructurePair(sequence=score_sequence, structure=score_structure)]
)
score_result = run_ligandmpnn_score(
    score_inputs,
    LigandMPNNScoringConfig(seed=42, return_logits=False),
)

score = score_result.scores[0]
print(f"Perplexity: {score.perplexity:.3f}")
print(f"Average log-likelihood: {score.avg_log_likelihood:.3f}")

**Input** — `LigandMPNNScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_structure_pairs</code> | <code>list[proto_tools.tools.inverse_folding.shared_data_models.SequenceStructurePair]</code> | required | List of sequence-structure pairs to score |

**Config** — `LigandMPNNScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on. |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Whether to include per-position logits in the output. |
| <code>scoring_mode</code> | <code>Literal['single_aa', 'autoregressive']</code> | <code>'single_aa'</code> | Use single-position probabilities or one seed-determined autoregressive order. |

**Output** — `InverseFoldingScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>scores</code> | <code>list[proto_tools.tools.inverse_folding.shared_data_models.InverseFoldingScoringMetrics]</code> | required | List of scoring outputs, one per input sequence-structure pair |

Running run_ligandmpnn_score [00:00]

Perplexity: 5.113
Average log-likelihood: -1.632


## Export Results

Designed sequences can be saved to standard file formats for downstream analysis. JSON format is convenient for preserving the full LigandMPNN metrics alongside the sequences, while FASTA format is useful for feeding results into other sequence analysis tools.

In [10]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="ligandmpnn_designs", export_path=output_dir, file_format="json")
score_result.export(name="ligandmpnn_scores", export_path=output_dir, file_format="json")
print(f"Results exported to {output_dir}")

Results exported to example_output
